In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("employee_attrition_2026_master.csv")

# Display first 5 rows
df.head()

,employee_id,age,generation,gender,department,job_level,is_manager,tenure_years,salary_usd,salary_vs_market_pct,...,burnout_score,manager_relationship_score,manager_1on1_per_month,engagement_score,promotions_last_2y,career_growth_score,recent_layoff_round,team_size,attrition_reason,attrition
0,EMP000000,26,Gen Z,Female,Marketing,Junior,0,5.5,70000,5.2,...,3.6,2,0,62,0,2,0,22,stayed,0
1,EMP000001,35,Millennial,Female,Sales,Junior,0,0.5,64000,-15.5,...,4.8,3,2,79,1,2,0,24,stayed,0
2,EMP000002,23,Gen Z,Male,HR,Senior,0,2.0,142000,-15.9,...,10.0,1,2,43,0,4,1,3,poor_management,1
3,EMP000003,20,Gen Z,Female,Sales,Manager,1,2.2,147000,2.6,...,4.3,1,4,64,1,3,0,24,stayed,0
4,EMP000004,40,Millennial,Female,Data,Senior,0,6.0,141000,11.3,...,2.9,1,1,92,0,2,0,5,stayed,0


In [2]:
# Dataset dimensions
print("Shape:", df.shape)

# Column names
print("\nColumns:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# Missing values
print("\nMissing Values:")
print(df.isnull().sum().sum())

# Duplicate records
print("\nDuplicate Records:")
print(df.duplicated().sum())

Shape: (45000, 29)

Columns:
['employee_id', 'age', 'generation', 'gender', 'department', 'job_level', 'is_manager', 'tenure_years', 'salary_usd', 'salary_vs_market_pct', 'work_arrangement', 'rto_mandate', 'days_in_office_required', 'commute_minutes', 'prefers_remote', 'flexibility_importance', 'ai_tools_adoption', 'weekly_hours', 'after_hours_work', 'burnout_score', 'manager_relationship_score', 'manager_1on1_per_month', 'engagement_score', 'promotions_last_2y', 'career_growth_score', 'recent_layoff_round', 'team_size', 'attrition_reason', 'attrition']

Data Types:
employee_id                       str
age                             int64
generation                        str
gender                            str
department                        str
job_level                         str
is_manager                      int64
tenure_years                  float64
salary_usd                      int64
salary_vs_market_pct          float64
work_arrangement                  str
rto_manda

In [3]:
# 1. Completeness
missing_values = df.isnull().sum().sum()
completeness = ((df.size - missing_values) / df.size) * 100

# 2. Uniqueness
duplicate_ids = df["employee_id"].duplicated().sum()
uniqueness = ((len(df) - duplicate_ids) / len(df)) * 100

# 3. Validity - Age should be between 18 and 100
invalid_age = ((df["age"] < 18) | (df["age"] > 100)).sum()

# 4. Consistency - Attrition reason should exist
# only when attrition = 1
inconsistent_attrition = (
    (df["attrition"] == 1) & 
    (df["attrition_reason"].isna())
).sum()

print("Completeness:", round(completeness, 2), "%")
print("Uniqueness:", round(uniqueness, 2), "%")
print("Invalid Age Records:", invalid_age)
print("Inconsistent Attrition Records:", inconsistent_attrition)

Completeness: 100.0 %
Uniqueness: 100.0 %
Invalid Age Records: 0
Inconsistent Attrition Records: 0


In [4]:
# Numerical data summary for accuracy review
print("Numerical Summary:")
display(df[["age", "tenure_years", "salary_usd"]].describe())

# Check date/time columns for timeliness
date_columns = [
    col for col in df.columns
    if "date" in col.lower() or "time" in col.lower()
]

print("\nDate/Time Columns:")
print(date_columns)

Numerical Summary:


,age,tenure_years,salary_usd
count,45000.000000,45000.000000,45000.000000
mean,34.240711,3.598353,115385.533333
std,8.547373,2.545593,50084.707992
min,20.000000,0.100000,35000.000000
25%,28.000000,1.700000,75000.000000
50%,34.000000,3.000000,103000.000000
75%,40.000000,4.800000,146000.000000
max,64.000000,25.000000,345000.000000



Date/Time Columns:
['rto_mandate']


In [5]:
quality_scorecard = pd.DataFrame([
    {
        "Dimension": "Completeness",
        "Check": "Missing values",
        "Result": "0 missing values",
        "Status": "Pass"
    },
    {
        "Dimension": "Accuracy",
        "Check": "Age and salary range review",
        "Result": "Values reviewed; source verification needed",
        "Status": "Needs Verification"
    },
    {
        "Dimension": "Consistency",
        "Check": "Attrition and reason relationship",
        "Result": "0 inconsistent records",
        "Status": "Pass"
    },
    {
        "Dimension": "Timeliness",
        "Check": "Date/time and freshness",
        "Result": "Freshness SLA not verified",
        "Status": "Needs Verification"
    },
    {
        "Dimension": "Uniqueness",
        "Check": "Duplicate employee IDs",
        "Result": "0 duplicate IDs",
        "Status": "Pass"
    },
    {
        "Dimension": "Validity",
        "Check": "Age between 18 and 100",
        "Result": "0 invalid records",
        "Status": "Pass"
    }
])

display(quality_scorecard)

,Dimension,Check,Result,Status
0,Completeness,Missing values,0 missing values,Pass
1,Accuracy,Age and salary range review,Values reviewed; source verification needed,Needs Verification
2,Consistency,Attrition and reason relationship,0 inconsistent records,Pass
3,Timeliness,Date/time and freshness,Freshness SLA not verified,Needs Verification
4,Uniqueness,Duplicate employee IDs,0 duplicate IDs,Pass
5,Validity,Age between 18 and 100,0 invalid records,Pass


In [6]:
%pip install great_expectations

     ---------------------------------------- 5.7/5.7 MB 5.7 MB/s eta 0:00:00
     ------------------------------------- 797.2/797.2 kB 12.5 MB/s eta 0:00:00
     ---------------------------------------- 3.8/3.8 MB 11.1 MB/s eta 0:00:00
     -------------------------------------- 134.9/134.9 kB 7.8 MB/s eta 0:00:00
     ---------------------------------------- 90.6/90.6 kB ? eta 0:00:00
     ---------------------------------------- 49.2/49.2 kB ? eta 0:00:00
     ---------------------------------------- 70.4/70.4 kB ? eta 0:00:00
     ------------------------------------- 490.4/490.4 kB 15.0 MB/s eta 0:00:00
     ---------------------------------------- 73.1/73.1 kB 3.9 MB/s eta 0:00:00
     ---------------------------------------- 118.1/118.1 kB ? eta 0:00:00
     ---------------------------------------- 36.6/36.6 MB 7.0 MB/s eta 0:00:00
     ---------------------------------------- 80.2/80.2 kB 4.7 MB/s eta 0:00:00
     ------------------------------------- 490.6/490.6 kB 10.2 MB/s e

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import great_expectations as gx

print("Great Expectations version:", gx.__version__)

Great Expectations version: 1.24.1


In [8]:
# Create 8 data quality checks using Pandas

checks = {
    "Employee ID not null": df["employee_id"].notna().all(),

    "Employee ID unique": df["employee_id"].is_unique,

    "Age between 18 and 100": df["age"].between(18, 100).all(),

    "Salary non-negative": (df["salary_usd"] >= 0).all(),

    "Tenure non-negative": (df["tenure_years"] >= 0).all(),

    "Attrition is binary": df["attrition"].isin([0, 1]).all(),

    "Department not null": df["department"].notna().all(),

    "Job level not null": df["job_level"].notna().all()
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nTotal checks:", len(checks))
print("Passed:", sum(checks.values()))
print("Failed:", len(checks) - sum(checks.values()))

Employee ID not null: PASS
Employee ID unique: PASS
Age between 18 and 100: PASS
Salary non-negative: PASS
Tenure non-negative: PASS
Attrition is binary: PASS
Department not null: PASS
Job level not null: PASS

Total checks: 8
Passed: 8
Failed: 0


In [9]:
# Create a separate test copy
test_df = df.copy()

# Error 1: Missing employee ID
test_df.loc[0, "employee_id"] = None

# Error 2: Invalid age
test_df.loc[1, "age"] = 150

# Error 3: Negative salary
test_df.loc[2, "salary_usd"] = -5000

print("Original dataset:", df.shape)
print("Test dataset:", test_df.shape)

print("\nInjected Errors:")
print("Missing Employee IDs:", test_df["employee_id"].isnull().sum())
print("Invalid Ages:", (test_df["age"] > 100).sum())
print("Negative Salaries:", (test_df["salary_usd"] < 0).sum())

Original dataset: (45000, 29)
Test dataset: (45000, 29)

Injected Errors:
Missing Employee IDs: 1
Invalid Ages: 1
Negative Salaries: 1


In [10]:
# Create Great Expectations DataFrame datasource
context = gx.get_context(mode="ephemeral")

data_source = context.data_sources.add_pandas(
    name="employee_data"
)

data_asset = data_source.add_dataframe_asset(
    name="employee_test_data"
)

batch_definition = data_asset.add_batch_definition_whole_dataframe(
    name="employee_batch"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": test_df}
)

# Define 3 expectations
expectations = [
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="employee_id"
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="age",
        min_value=18,
        max_value=100
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="salary_usd",
        min_value=0
    )
]

# Run validation
for expectation in expectations:
    result = batch.validate(expectation)
    print(expectation.__class__.__name__, ":", result.success)

Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 258.04it/s] 


ExpectColumnValuesToNotBeNull : False


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 200.26it/s]


ExpectColumnValuesToBeBetween : False


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 192.86it/s]

ExpectColumnValuesToBeBetween : False


In [11]:
# Add remaining 5 expectations
more_expectations = [
    gx.expectations.ExpectColumnValuesToBeUnique(
        column="employee_id"
    ),
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="tenure_years",
        min_value=0
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="attrition",
        value_set=[0, 1]
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="department"
    ),
    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="job_level"
    )
]

# Validate remaining expectations
for expectation in more_expectations:
    result = batch.validate(expectation)
    print(type(expectation).__name__, ":", result.success)

Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 153.84it/s]


ExpectColumnValuesToBeUnique : True


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 454.53it/s]


ExpectColumnValuesToBeBetween : True


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 588.41it/s]


ExpectColumnValuesToBeInSet : True


Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 533.26it/s] 


ExpectColumnValuesToNotBeNull : True


Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 285.05it/s]

ExpectColumnValuesToNotBeNull : True


In [12]:
# Final validation summary
print("GREAT EXPECTATIONS SUMMARY")
print("--------------------------")
print("Total Expectations:", 8)
print("Passed:", 5)
print("Failed:", 3)
print("Errors intentionally injected:", 3)

print("\nConclusion:")
print("Great Expectations successfully detected all 3 data quality errors.")

GREAT EXPECTATIONS SUMMARY
--------------------------
Total Expectations: 8
Passed: 5
Failed: 3
Errors intentionally injected: 3

Conclusion:
Great Expectations successfully detected all 3 data quality errors.


In [13]:
# Combine all 8 expectations
all_expectations = expectations + more_expectations

# Store validation results
validation_results = []

for exp in all_expectations:
    result = batch.validate(exp)

    validation_results.append({
        "Expectation": type(exp).__name__,
        "Status": "PASS" if result.success else "FAIL"
    })

# Convert results into a DataFrame
report_df = pd.DataFrame(validation_results)

# Generate HTML report
html_report = report_df.to_html(
    index=False,
    border=1
)

with open("great_expectations_report.html", "w", encoding="utf-8") as file:
    file.write("<h1>Great Expectations Validation Report</h1>")
    file.write(html_report)

print("HTML validation report created successfully!")

Calculating Metrics: 100%|██████████| 8/8 [00:00<00:00, 396.62it/s] 

HTML validation report created successfully!


In [14]:
pii_classification = pd.DataFrame([
    ["employee_id", "Personal Identifier", "Tokenisation"],
    ["age", "Personal Data", "Masking"],
    ["generation", "Personal Data", "Restricted Access"],
    ["gender", "Personal Data", "Restricted Access"],
    ["salary_usd", "Confidential Employment Data", "Restricted Access"],
    ["salary_vs_market_pct", "Confidential Employment Data", "Restricted Access"],
    ["burnout_score", "Confidential Employee Data", "Restricted Access"],
    ["manager_relationship_score", "Confidential Employee Data", "Restricted Access"],
    ["engagement_score", "Confidential Employee Data", "Restricted Access"],
    ["career_growth_score", "Confidential Employee Data", "Restricted Access"],
    ["attrition_reason", "Confidential Employment Data", "Restricted Access"],
    ["department", "Internal Business Data", "Role-Based Access"],
    ["job_level", "Internal Business Data", "Role-Based Access"]
], columns=["Column", "Classification", "Protection Strategy"])

display(pii_classification)

,Column,Classification,Protection Strategy
0,employee_id,Personal Identifier,Tokenisation
1,age,Personal Data,Masking
2,generation,Personal Data,Restricted Access
3,gender,Personal Data,Restricted Access
4,salary_usd,Confidential Employment Data,Restricted Access
5,salary_vs_market_pct,Confidential Employment Data,Restricted Access
6,burnout_score,Confidential Employee Data,Restricted Access
7,manager_relationship_score,Confidential Employee Data,Restricted Access
8,engagement_score,Confidential Employee Data,Restricted Access
9,career_growth_score,Confidential Employee Data,Restricted Access


In [15]:
from ruamel.yaml import YAML

yaml = YAML(typ="safe")

with open("data_contract_employee_attrition.yaml", "r") as file:
    contract = yaml.load(file)

print("YAML Validation: Successful")
print("Contract ID:", contract["id"])
print("Producer:", contract["info"]["producer"])
print("Consumer:", contract["info"]["consumer"])
print("Data Models:", list(contract["models"].keys()))

YAML Validation: Successful
Contract ID: employee-attrition-v1
Producer: Anusha
Consumer: BI-dashboard-team
Data Models: ['employee_attrition']
